# Project: kNN and the curse of dimensionality 📏
k-nearest neighbours is the simplest classifier: vote among the closest training points. It works beautifully on handwritten digits and then falls apart when you add irrelevant dimensions, because **in high dimensions everything is far from everything**. Measure it.

Topic: [[k-Nearest Neighbors]], [[Dimensionality Reduction]] · guide note: [[Project - kNN and the Curse of Dimensionality]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
X, y = load_digits(return_X_y=True); X = X / 16
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## 1. kNN from scratch (vectorised)
Squared Euclidean distances between every test and training point, take the `k` nearest, majority vote (`np.bincount(...).argmax()`, ties → smallest label).

In [ ]:
def knn_predict(X_train, y_train, X_test, k=3):
    # TODO 1: distance matrix, k nearest, majority vote
    raise NotImplementedError  # TODO 1

In [ ]:
check("kNN", lambda: list(knn_predict(np.array([[0.0], [1], [10], [11]]), np.array([0, 0, 1, 1]), np.array([[0.4], [10.6]]), k=3)) == [0, 1],
      "1-D toy: 0.4 is near the 0s, 10.6 near the 1s.")
acc = {}
if ready("kNN"):
    for k in (1, 3, 7): acc[k] = np.mean(knn_predict(X_tr, y_tr, X_te, k) == y_te); print(f"k={k}: test accuracy {acc[k]:.3f}")

## 2. Everything is far away
Sample `n` uniform points in the unit cube `[0,1]^dim` and one query point. Return `min distance / max distance` from the query. In 2-D the nearest point is much closer than the farthest; as `dim` grows the ratio → 1 and "nearest" stops meaning anything.

In [ ]:
def distance_ratio(dim, n=1000, seed=0):
    # TODO 2: nearest/farthest distance ratio for a random query
    raise NotImplementedError  # TODO 2

In [ ]:
check("distance ratio", lambda: distance_ratio(2) < 0.05 and distance_ratio(1000) > 0.8, "2-D: tiny ratio; 1000-D: close to 1.")
if ready("kNN", "distance ratio"):
    dims = [1, 2, 5, 10, 50, 100, 500, 1000]; r = [np.mean([distance_ratio(d, seed=s) for s in range(5)]) for d in dims]
    plt.semilogx(dims, r, "o-"); plt.xlabel("dimensions"); plt.ylabel("nearest / farthest distance"); plt.show()
    rng = np.random.default_rng(0); noisy = {}
    for extra in (0, 64, 256, 1024):                          # append irrelevant random features
        Ntr, Nte = rng.random((len(X_tr), extra)), rng.random((len(X_te), extra))
        noisy[extra] = np.mean(knn_predict(np.hstack([X_tr, Ntr]), y_tr, np.hstack([X_te, Nte]), 3) == y_te)
        print(f"+{extra:>4} noise features: accuracy {noisy[extra]:.3f}")
    check("curse observed", lambda: noisy[1024] < noisy[0] - 0.3, "1024 irrelevant features should cost kNN more than 30 accuracy points.")

<details><summary>▶ Solution</summary>

```python
def knn_predict(X_train, y_train, X_test, k=3):
    d = ((X_test[:, None, :] - X_train[None, :, :]) ** 2).sum(-1)
    nn = np.argsort(d, axis=1)[:, :k]
    return np.array([np.bincount(y_train[row]).argmax() for row in nn])
```

```python
def distance_ratio(dim, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.random((n, dim)); q = rng.random(dim)
    d = np.sqrt(((X - q) ** 2).sum(1))
    return float(d.min() / d.max())
```
</details>

## Stretch goals
- Fix the noisy version with PCA to 30 dimensions before kNN ([[PCA]]). How much accuracy comes back?
- Would logistic regression suffer as much from the noise features? Try it: why the difference?
- Plot the histogram of pairwise distances in 2, 100 and 1000 dimensions: concentration of measure.